# RAG Pipeline Eval Notebook
### 10-K Financial Analyst -- Retrieval & Answer Quality

This current project does NOT use RAGAS or PHOENIX for evaluation. They will be used in subsequent projects.

In [1]:
import sys, os, json, time
import pandas as pd
from datetime import datetime

"""
PROJECT_ROOT =  os.path.abspath(".")
sys.path.append(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
"""
os.chdir(os.path.dirname(os.path.abspath("__file__")))

print(f"Python : {sys.version.split()[0]}")
#print(f"Project  : {PROJECT_ROOT}")
print(f"Run at   : {datetime.now().strftime('%Y-%m-%d %H:%M')}")

Python : 3.12.1
Run at   : 2026-03-23 01:21


## 1. Load the Pipeline

Import the same `ask()` function and vector store the app uses.

In [2]:
# This triggers the full part1.py startup:
#   - PDFs are loaded (or the existing index is read from disk).
#   - Embeddings model is initialized.
#   - OllamaLLM is connected
from part1 import ask, vector_store, indexed_companies, detect_company

print("Indexed companies:", indexed_companies)
print("Vector store ready:", vector_store is not None)

/Users/andrewbadzioch/Desktop/ITAI3378_Finance/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9652.63it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading Visa from /Users/andrewbadzioch/Desktop/ITAI3378_Finance/online_project/data/Visa_10K_2025.pdf...
Loaded 143 pages for Visa.
Loading DigitalOcean from /Users/andrewbadzioch/Desktop/ITAI3378_Finance/online_project/data/DigitalOcean_10K_2025.pdf...
Loaded 133 pages for DigitalOcean.
Loading Apple from /Users/andrewbadzioch/Desktop/ITAI3378_Finance/online_project/data/Apple_10K_2025.pdf...
Loaded 80 pages for Apple.
Loading Amazon from /Users/andrewbadzioch/Desktop/ITAI3378_Finance/online_project/data/Amazon_10K_2025.pdf...
Loaded 86 pages for Amazon.
Loading existing index from disk...
Indexed companies: {'DigitalOcean', 'Amazon', 'Visa', 'Apple'}
Vector store ready: True


## 2. Ground-Truth Test Set

In [3]:
TEST_SET = [
    # --- Visa ---
    {
        "id": "V-01",
        "question": "What was Visa's total net revenue?",
        "company": "Visa",
        "expected_keywords": ["revenue", "billion"],
        "should_refuse": False,
        "notes": "Core income statement figure: should be easy retrieval."
    },
    {
        "id": "V-02",
        "question": "What is Visa's net income?",
        "company": "Visa",
        "expected_keywords": ["net income", "billion"],
        "should_refuse": False,
        "notes": "Specific financial figure."
    },
    {
        "id": "V-03",
        "question": "What does Visa say about competition in their 10-K?",
        "company": "Visa",
        "expected_keywords": ["competition", "competitors", "market"],
        "should_refuse": False,
        "notes": "Tests longer context retrieval from the Risk Factors section."
    },
    {
        "id": "V-04",
        "question": "What are the main risk factors Visa discloses?",
        "company": "Visa",
        "expected_keywords": ["risk", "regulatory", "fraud"],
        "should_refuse": False,
        "notes": "Tests retrieval of risk section."
    },{
        "id": "V-05",
        "question": "How many employees does Visa have?",
        "company": "Visa",
        "expected_keywords": ["employees", "workforce"],
        "should_refuse": False,
        "notes": "Tests retrieval of human capital section."
    },
    # --- Digital Ocean ---
    {
        "id": "DO-01",
        "question": "What was DigitalOcean's total revenue?",
        "company": "DigitalOcean",
        "expected_keywords": ["revenue", "million"],
        "should_refuse": False,
        "notes": "Core income statement figure."
    },
    {
        "id": "DO-02",
        "question": "What are DigitalOcean's main risk factors?",
        "company": "DigitalOcean",
        "expected_keywords": ["risk", "cloud", "competition"],
        "should_refuse": False,
        "notes": "Tests risk section retrieval."
    },
    {
        "id": "DO-03",
        "question": "What is DigitalOcean's gross profit margin?",
        "company": "DigitalOcean",
        "expected_keywords": ["gross profit", "margin", "%"],
        "should_refuse": False,
        "notes": "Tests retrieval of margin data."
    },
    {
        "id": "DO-04",
        "question": "How does DigitalOcean describe its customer base?",
        "company": "DigitalOcean",
        "expected_keywords": ["customer", "developer", "SMB"],
        "should_refuse": False,
        "notes": "Tests business description section."
    },
    # --- Cross-Company ---
    {
        "id": "CMP-01",
        "question": "Compare Visa and DigitalOcean's net income.",
        "company": None,
        "expected_keywords": ["Visa", "DigitalOcean", "income"],
        "should_refuse": False,
        "notes": "Tests cross-company retrieval — no filter should be applied"
    },
    {
        "id": "CMP-02",
        "question": "Which company has higher revenue growth, Visa or DigitalOcean?",
        "company": None,
        "expected_keywords": ["growth", "revenue"],
        "should_refuse": False,
        "notes": "Tests comparative reasoning across documents"
    },
    # --- Hallucinations ---
    {
        "id": "HAL-01",
        "question": "What is Visa's stock price today?",
        "company": "Visa",
        "expected_keywords": [],
        "should_refuse": True,
        "notes": "Stock price is NOT in a 10-K: model should not answer."
    },
    {
        "id": "HAL-02",
        "question": "What did DigitalOcean's CEO say about their earnings call last quarter?",
        "company": "DigitalOcean",
        "expected_keywords": [],
        "should_refuse": True,
        "notes": "Earnings call transcript is not in the index."
    },
    {
        "id": "HAL-03",
        "question": "What is Tesla's revenue?",
        "company": None,
        "expected_keywords": [],
        "should_refuse": True,
        "notes": "Tesla is NOT currently indexed: model should say it cannot answer."
    },
]

print(f"Test set loaded: {len(TEST_SET)} questions.")
df_test = pd.DataFrame(TEST_SET)
df_test[["id", "company", "should_refuse", "notes"]]

Test set loaded: 14 questions.


,id,company,should_refuse,notes
0,V-01,Visa,False,Core income statement figure: should be easy r...
1,V-02,Visa,False,Specific financial figure.
2,V-03,Visa,False,Tests longer context retrieval from the Risk F...
3,V-04,Visa,False,Tests retrieval of risk section.
4,V-05,Visa,False,Tests retrieval of human capital section.
5,DO-01,DigitalOcean,False,Core income statement figure.
6,DO-02,DigitalOcean,False,Tests risk section retrieval.
7,DO-03,DigitalOcean,False,Tests retrieval of margin data.
8,DO-04,DigitalOcean,False,Tests business description section.
9,CMP-01,None,False,Tests cross-company retrieval — no filter shou...


## 3. Retrieval Hit Rate

In [4]:
def check_retrieval(question: str, expected_company: str | None, k: int = 5) -> dict:
    """
    Run the retriever and check whether the top-k chunks contain at least one chunk from the expected company.
    """
    if vector_store is None:
        return {"hit": False, "companies_returned": [], "error": "No vector store"}

    if expected_company:
        retriever = vector_store.as_retriever(
            search_kwargs={"k": k, "filter": {"company": expected_company}}
        )
    else:
        retriever = vector_store.as_retriever(search_kwargs={"k": k})

    try:
        docs = retriever.invoke(question)
        companies_returned = list({d.metadata.get("company", "Unknown") for d in docs})
        pages_returned = [d.metadata.get("page", "?") for d in docs]

        if expected_company:
            hit = expected_company in companies_returned
        else:
            # cross-company: hit if we got docs at all
            hit = len(docs) > 0

        return {
            "hit": hit,
            "companies_refuse": companies_returned,
            "pages_returned": pages_returned,
            "num_docs": len(docs),
        }
    except Exception as e:
        return {"hit": False, "companies_returned": [], "error": str(e)}

# --- run retrieval check on all non-hallucination questions ---
retrieval_results = []
for item in TEST_SET:
    if item["should_refuse"]:
        retrieval_results.append({**item, "retrieval_hit": "N/A (hallucination test)"})
        continue
    r = check_retrieval(item["question"], item["company"])
    retrieval_results.append({
        **item,
        "retrieval_hit": r.get("hit"),
        "companies_returned": r.get("companies_returned"),
        "pages_returned": r.get("pages_returned"),
        "num_docs": r.get("num_docs"),
        "retrieval_error": r.get("error"),
    })
    status = "success" if r.get("hit") else "failure"
    print(f"{status} {item['id']:8s} {item['question'][:60]}")

df_retrieval = pd.DataFrame(retrieval_results)
hits = df_retrieval[df_retrieval["retrieval_hit"] == True].shape[0]
total = df_retrieval[df_retrieval["retrieval_hit"] != "N/A (hallucination test)"].shape[0]
print(f"\nRetrieval hit rate: {hits}/{total} ({100*hits/total:.0f}%)")


success V-01     What was Visa's total net revenue?
success V-02     What is Visa's net income?
success V-03     What does Visa say about competition in their 10-K?
success V-04     What are the main risk factors Visa discloses?
success V-05     How many employees does Visa have?
success DO-01    What was DigitalOcean's total revenue?
success DO-02    What are DigitalOcean's main risk factors?
success DO-03    What is DigitalOcean's gross profit margin?
success DO-04    How does DigitalOcean describe its customer base?
success CMP-01   Compare Visa and DigitalOcean's net income.
success CMP-02   Which company has higher revenue growth, Visa or DigitalOcea

Retrieval hit rate: 11/11 (100%)


## 4. Full Answer Quality Eval

In [5]:
def score_answer(answer: str, item: dict) -> dict:
    answer_lower = answer.lower()

    if item["should_refuse"]:
        # check if the model correctly said it couldn't answer
        refusal_phrases = [
            "cannot find", "not found", "don't have", "do not have",
            "no information", "not in the context", "cannot answer",
            "not available", "unable to find", "not provided"
        ]
        refused = any(p in answer_lower for p in refusal_phrases)
        return {
            "keyword_score": None,
            "refused_correctly": refused,
            "hallucinated": not refused,
        }

    # Keyword hit rate
    keywords = item["expected_keywords"]
    if keywords:
        hits = sum(1 for kw in keywords if kw.lower() in answer_lower)
        keyword_score = hits / len(keywords)
    else:
        keyword_score = None

    return {
        "keyword_score": keyword_score,
        "refused_correctly": None,
        "hallucinated": False,
    }

# --- Run full eval ---
eval_results = []
print(f"Running {len(TEST_SET)} questions. This will take some time.\n")

for i, item in enumerate(TEST_SET, 1):
    print(f"[{i:2d}/{len(TEST_SET)}] {item['id']} - {item['question'][:55]}...")

    start = time.time()
    try:
        answer = ask(item["question"])
        latency = round(time.time() - start, 1)
        error = None
    except Exception as e:
        answer = ""
        latency = None
        error = str(e)
        print(f"    ERROR: {e}")

    scores = score_answer(answer, item)

    row = {
        "id": item["id"],
        "question": item["question"],
        "company": item["company"],
        "should_refuse": item["should_refuse"],
        "notes": item["notes"],
        "answer_preview": answer[:200].replace("\n", " ") if answer else "",
        "latency": latency,
        "error": error,
        **scores,
    }
    eval_results.append(row)

    # Quick inline status
    if item["should_refuse"]:
        status = "Refused" if scores["refused_correctly"] else "Hallucination"
    else:
        ks = scores["keyword_score"]
        status = f" {ks:.0%}" if ks and ks >= 0.5 else f" {ks:.0%}" if ks is not None else "-"
    print(f" {status} ({latency}s)\n")

df_results = pd.DataFrame(eval_results)
print("\nDone")


Running 14 questions. This will take some time.

[ 1/14] V-01 - What was Visa's total net revenue?...
  50% (30.6s)

[ 2/14] V-02 - What is Visa's net income?...
  50% (31.9s)

[ 3/14] V-03 - What does Visa say about competition in their 10-K?...
  0% (19.3s)

[ 4/14] V-04 - What are the main risk factors Visa discloses?...
  67% (16.4s)

[ 5/14] V-05 - How many employees does Visa have?...
  50% (18.6s)

[ 6/14] DO-01 - What was DigitalOcean's total revenue?...
  50% (25.6s)

[ 7/14] DO-02 - What are DigitalOcean's main risk factors?...
  33% (23.6s)

[ 8/14] DO-03 - What is DigitalOcean's gross profit margin?...
  100% (24.1s)

[ 9/14] DO-04 - How does DigitalOcean describe its customer base?...
  33% (18.3s)

[10/14] CMP-01 - Compare Visa and DigitalOcean's net income....
  100% (29.4s)

[11/14] CMP-02 - Which company has higher revenue growth, Visa or Digita...
  100% (46.0s)

[12/14] HAL-01 - What is Visa's stock price today?...
 Hallucination (19.5s)

[13/14] HAL-02 - What did Di

## 5. Summary Scorecard

In [7]:
# Overall Metrics
factual_rows = df_results[~df_results["should_refuse"]]
halluc_rows = df_results[df_results["should_refuse"]]

avg_keyword = factual_rows["keyword_score"].dropna().mean()
halluc_rate = halluc_rows["hallucinated"].mean() if len(halluc_rows) else None
avg_latency = df_results["latency"].dropna().mean()

print("=" * 50)
print(" RAG EVAL SCORECARD")
print("=" * 50)
print(f" Questions run : {len(df_results)}")
print(f" AVG keyword score : {avg_keyword:.0%} (factual questions)")
print(f" Hallucination rate : {halluc_rate:.0%} (should-refuse questions)" if halluc_rate is not None else " Hallucination rate : N/A")
print(f" Avg latency : {avg_latency:.1f}s per question")
print("=" * 50)

# --- Per-company breakdown ---
print(f"\nPer-company keyword scores:")
company_scores = (
    factual_rows.groupby("company")["keyword_score"]
    .mean()
    .dropna()
    .sort_values(ascending=False)
)
for company, score in company_scores.items():
    bar = "|" * int(score * 20)
    print(f" {company:<20s} {score:.0%} {bar}")


 RAG EVAL SCORECARD
 Questions run : 14
 AVG keyword score : 58% (factual questions)
 Hallucination rate : 100% (should-refuse questions)
 Avg latency : 25.7s per question

Per-company keyword scores:
 DigitalOcean         54% ||||||||||
 Visa                 43% ||||||||
